# Bachelor Prototype

Thank you for participating.

This prototype contains five interactive programming exercises. The programming language is Python. The aim is to test the design of the exercises and to facilitate reasoning about the chosen programming concepts, not to test your capabilities as a programmer.

While solving the tasks, please focus on saying what you notice, why you choose a line or group, what feels uncertain, what works well, and what works less well. After the tasks, you will be asked to comment more directly on the design of the exercises.

You may use the hint, solution, and walkthrough buttons when they become available. They are part of the prototype, so using them is not a failure.

There are five tasks in total:
1. three tasks about execution order and sequential processes;
2. two tasks about program structure, decomposition, and abstraction.

In [7]:

import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
import re


def _show(widget):
    widget.layout.display = ''


def normalize_list_answer(text):
    return [part.strip() for part in text.split(',') if part.strip()]


def normalize_order_alternatives(correct_order):
    """Allow tasks to define either one correct order or several acceptable orders."""
    if not correct_order:
        return []
    if all(isinstance(item, str) for item in correct_order):
        return [correct_order]
    return correct_order


def create_order_task(task_id, correct_order, hint_md, solution_md, walkthrough_md, n_lines):
    answer_input = widgets.Text(
        value='',
        placeholder='',
        description='Order:',
        layout=widgets.Layout(width='360px')
    )

    check_button = widgets.Button(description='Check answer', button_style='success')
    hint_button = widgets.Button(description='Show hint', layout=widgets.Layout(display='none'))
    walkthrough_button = widgets.Button(description='Show walkthrough', layout=widgets.Layout(display='none'))
    solution_button = widgets.Button(description='Show solution', layout=widgets.Layout(display='none'))

    feedback = widgets.Output()
    support = widgets.Output()
    wrong_attempts = {'count': 0}
    acceptable_orders = normalize_order_alternatives(correct_order)
    expected_line_numbers = sorted(acceptable_orders[0])

    def on_check_clicked(_):
        with feedback:
            clear_output()
            user_answer = normalize_list_answer(answer_input.value)

            if len(user_answer) != n_lines:
                print(f'Please enter exactly {n_lines} line numbers, separated by commas.')
                return

            if sorted(user_answer) != expected_line_numbers:
                print('Please use each line number exactly once.')
                return

            if user_answer in acceptable_orders:
                display(Markdown('**Correct.**'))
            else:
                wrong_attempts['count'] += 1
                if wrong_attempts['count'] == 1:
                    _show(hint_button)
                    display(Markdown('**Not quite.** You can now choose to show a hint.'))
                else:
                    _show(hint_button)
                    _show(walkthrough_button)
                    _show(solution_button)
                    display(Markdown('**Still not quite.** You can now choose to see the walkthrough or the solution.'))

    def show_hint(_):
        with support:
            clear_output()
            display(Markdown(hint_md))

    def show_solution(_):
        with support:
            clear_output()
            display(Markdown(solution_md))

    def show_walkthrough(_):
        with support:
            clear_output()
            display(Markdown(walkthrough_md))

    check_button.on_click(on_check_clicked)
    hint_button.on_click(show_hint)
    solution_button.on_click(show_solution)
    walkthrough_button.on_click(show_walkthrough)

    display(widgets.VBox([
        answer_input,
        widgets.HBox([check_button, hint_button, walkthrough_button, solution_button]),
        feedback,
        support
    ]))


def normalize_blocks(text):
    groups = text.split(';')
    normalized = []
    for group in groups:
        items = [part.strip() for part in group.split(',') if part.strip()]
        if items:
            normalized.append(items)
    return normalized


def same_block_membership(user_blocks, correct_blocks):
    """Compare groups by membership, not by the order of line numbers inside each group."""
    return [set(block) for block in user_blocks] == [set(block) for block in correct_blocks]


def create_block_task(correct_blocks, hint_md, solution_md, walkthrough_md, all_lines):
    answer_input = widgets.Text(
        value='',
        placeholder='',
        description='Blocks:',
        layout=widgets.Layout(width='460px')
    )

    check_button = widgets.Button(description='Check answer', button_style='success')
    hint_button = widgets.Button(description='Show hint', layout=widgets.Layout(display='none'))
    walkthrough_button = widgets.Button(description='Show walkthrough', layout=widgets.Layout(display='none'))
    solution_button = widgets.Button(description='Show solution', layout=widgets.Layout(display='none'))

    feedback = widgets.Output()
    support = widgets.Output()
    wrong_attempts = {'count': 0}

    def on_check_clicked(_):
        with feedback:
            clear_output()
            user_blocks = normalize_blocks(answer_input.value)

            if len(user_blocks) != len(correct_blocks):
                print(f'Please enter exactly {len(correct_blocks)} groups separated by semicolons.')
                return

            flat = [item for group in user_blocks for item in group]
            if sorted(flat) != sorted(all_lines) or len(flat) != len(set(flat)):
                print('Please use each line number exactly once.')
                return

            if same_block_membership(user_blocks, correct_blocks):
                display(Markdown('**Correct.**'))
            else:
                wrong_attempts['count'] += 1
                if wrong_attempts['count'] == 1:
                    _show(hint_button)
                    display(Markdown('**Not quite.** You can now choose to show a hint.'))
                else:
                    _show(hint_button)
                    _show(walkthrough_button)
                    _show(solution_button)
                    display(Markdown('**Still not quite.** You can now choose to see the walkthrough or the solution.'))

    def show_hint(_):
        with support:
            clear_output()
            display(Markdown(hint_md))

    def show_solution(_):
        with support:
            clear_output()
            display(Markdown(solution_md))

    def show_walkthrough(_):
        with support:
            clear_output()
            display(Markdown(walkthrough_md))

    check_button.on_click(on_check_clicked)
    hint_button.on_click(show_hint)
    solution_button.on_click(show_solution)
    walkthrough_button.on_click(show_walkthrough)

    display(widgets.VBox([
        answer_input,
        widgets.HBox([check_button, hint_button, walkthrough_button, solution_button]),
        feedback,
        support
    ]))


def parse_line_groups(text):
    # Accept formats like "1,2 and 4,5", "1,2; 4,5", or "Repeated lines: 1,2 and 4,5".
    text = text.lower().replace('&', 'and').replace(';', 'and')
    raw_groups = [part.strip() for part in text.split('and') if part.strip()]
    groups = []

    for group in raw_groups:
        numbers = re.findall(r'\d+', group)
        if len(numbers) >= 2:
            groups.append(tuple(sorted(numbers)))
    return sorted(groups)


def normalize_expected_group_alternatives(expected_group_alternatives):
    normalized = []
    for alternative in expected_group_alternatives:
        normalized.append(sorted([tuple(sorted(group)) for group in alternative]))
    return normalized


def parse_inputs(text):
    # Accept formats like "name, price", "Inputs: title and price", or "product model + price".
    return [token.lower() for token in re.findall(r'[A-Za-z_]\w*', text)]


def input_answer_is_reasonable(tokens):
    # Keep this deliberately flexible. The explanation matters more than exact wording.
    has_price = any('price' in token for token in tokens)
    has_name_like_input = any(
        keyword in token
        for token in tokens
        for keyword in ['name', 'title', 'product', 'item', 'model']
    )
    return has_price and has_name_like_input


def create_abstraction_task(expected_group_alternatives, hint_md, solution_md, walkthrough_md):
    shared_style = {'description_width': '150px'}
    shared_layout = widgets.Layout(width='500px')

    lines_input = widgets.Text(
        description='Repeated lines:',
        style=shared_style,
        layout=shared_layout
    )

    name_input = widgets.Text(
        description='Function name:',
        style=shared_style,
        layout=shared_layout
    )

    inputs_input = widgets.Text(
        description='Function parameters:',
        style=shared_style,
        layout=shared_layout
    )

    check_button = widgets.Button(description='Check answer', button_style='success')
    hint_button = widgets.Button(description='Show hint', layout=widgets.Layout(display='none'))
    walkthrough_button = widgets.Button(description='Show walkthrough', layout=widgets.Layout(display='none'))
    solution_button = widgets.Button(description='Show solution', layout=widgets.Layout(display='none'))

    feedback = widgets.Output()
    support = widgets.Output()
    wrong_attempts = {'count': 0}

    expected_alternatives = normalize_expected_group_alternatives(expected_group_alternatives)

    def acceptable_name(name):
        name = name.strip()
        if not name:
            return False
        if ' ' in name or '-' in name:
            return False
        if len(name) < 3:
            return False
        # Keep this deliberately permissive. The explanation matters more than an exact name.
        return bool(re.match(r'^[A-Za-z_]\w*$', name))

    def on_check_clicked(_):
        with feedback:
            clear_output()
            groups = parse_line_groups(lines_input.value)
            name = name_input.value.strip()
            input_tokens = parse_inputs(inputs_input.value)

            if not lines_input.value.strip() or not name or not inputs_input.value.strip():
                print('Please answer all three parts.')
                return

            line_check = groups in expected_alternatives
            name_check = acceptable_name(name)
            inputs_check = input_answer_is_reasonable(input_tokens)

            if line_check and name_check and inputs_check:
                display(Markdown('**Reasonable answer.** You identified the repeated structure and suggested a plausible abstraction.'))
            else:
                wrong_attempts['count'] += 1
                if wrong_attempts['count'] == 1:
                    _show(hint_button)
                    display(Markdown('**Not quite.** One part of your answer does not match the intended function boundary clearly enough. You can now choose to show a hint.'))
                else:
                    _show(hint_button)
                    _show(walkthrough_button)
                    _show(solution_button)
                    display(Markdown('**Still not quite.** You can now choose to see the walkthrough or one possible solution.'))

    def show_hint(_):
        with support:
            clear_output()
            display(Markdown(hint_md))

    def show_solution(_):
        with support:
            clear_output()
            display(Markdown(solution_md))

    def show_walkthrough(_):
        with support:
            clear_output()
            display(Markdown(walkthrough_md))

    check_button.on_click(on_check_clicked)
    hint_button.on_click(show_hint)
    solution_button.on_click(show_solution)
    walkthrough_button.on_click(show_walkthrough)

    display(widgets.VBox([
        lines_input,
        name_input,
        inputs_input,
        widgets.HBox([check_button, hint_button, walkthrough_button, solution_button]),
        feedback,
        support
    ]))


# Concept 1: Sequential Processes

In these tasks, the code lines are shown in the wrong order.

Your task is to decide an execution order where each line can run without causing a variable error.

Write your answer as a comma-separated sequence of line numbers, for example:

`2,1,3,4,6,5`


### Task 1.1: Finding the starting point

This program builds a simple price label for a product. Arrange the lines so the program runs without causing an error.

1. `discount = price * 0.10`  
2. `label = "Final price: " + str(final_price)`  
3. `price = base_price + fee`  
4. `final_price = price - discount`  
5. `fee = base_price * 0.05`  
6. `base_price = 100`


In [8]:

create_order_task(
    task_id='vd',
    correct_order=['6', '5', '3', '1', '4', '2'],
    n_lines=6,
    hint_md="""**Hint:** Start with the line that can run without needing any earlier values, then follow the dependencies.""",
    solution_md="""**Solution:** `6,5,3,1,4,2`""",
    walkthrough_md="""**Walkthrough:**

1. `base_price = 100` must come first, because it creates `base_price`.
2. `fee = base_price * 0.05` can now run, because `base_price` exists.
3. `price = base_price + fee` can now run, because both `base_price` and `fee` exist.
4. `discount = price * 0.10` can now run, because `price` exists.
5. `final_price = price - discount` can now run, because both `price` and `discount` exist.
6. `label = "Final price: " + str(final_price)` comes last, because it depends on `final_price`.

**Takeaway:** The execution order is constrained by variable dependencies: a line can only run when the variables it uses already exist.""",
)


### Task 1.2: What happens inside the loop?

This program doubles each number in a small list and adds the doubled values to a running total. Arrange the lines so the program correctly computes and prints the total.

At least one line belongs inside the loop and should be indented in the final program.

1. `total = total + doubled`  
2. `numbers = [1, 2, 3]`  
3. `print(total)`  
4. `doubled = n * 2`  
5. `total = 0`  
6. `for n in numbers:`


In [9]:

create_order_task(
    task_id='loop',
    correct_order=[['2', '5', '6', '4', '1', '3'], ['5', '2', '6', '4', '1', '3']],
    n_lines=6,
    hint_md="""**Hint:** Which setup lines must happen before the loop starts? And which lines depend on the loop initialisation?""",
    solution_md="""**One valid solution:** `2,5,6,4,1,3`

`5,2,6,4,1,3` is also valid, because `total = 0` and `numbers = [1, 2, 3]` are both setup lines that do not depend on each other.""",
    walkthrough_md="""**Walkthrough:**

1. `numbers = [1, 2, 3]` creates the data the loop will use.
2. `total = 0` initializes the accumulator.
3. `for n in numbers:` starts the loop. It uses `numbers`, so the list must already exist.
4. `doubled = n * 2` belongs inside the loop. It creates a doubled value for the current `n`.
5. `total = total + doubled` also belongs inside the loop. It updates the accumulator, so both `total` and `doubled` must already exist.
6. `print(total)` comes after the loop, because the total should be printed after all numbers have been processed.

**Takeaway:** The setup must happen before the loop starts, and the loop body also has its own execution order. Here, `doubled` must be created before it can be added to `total`.""",
)


### Task 1.3: Building results through dependencies

This program builds two totals step by step and prints the final label. Arrange the lines so the program runs without causing an error.

As you solve the task, notice whether any parts of the program have a similar structure. Could repeated logic be made into a function instead?

1. `first_label = "First total: " + str(first_total)`  
2. `second_total = second_subtotal * 1.25`  
3. `first_subtotal = price + shipping`  
4. `print(second_label)`  
5. `second_label = first_label + " | Second total: " + str(second_total)`  
6. `price = 100`  
7. `second_subtotal = first_total + extra_item`  
8. `shipping = price * 0.20`  
9. `first_total = first_subtotal * 1.25`  
10. `extra_item = len(first_label)`

In [10]:

create_order_task(
    task_id='repeated_steps',
    correct_order=['6', '8', '3', '9', '1', '10', '7', '2', '5', '4'],
    n_lines=10,
    hint_md="""**Hint:** Start with the variable that does not depend on any other variables, then follow the chain of dependencies.""",
    solution_md="""**Solution:** `6,8,3,9,1,10,7,2,5,4`""",
    walkthrough_md="""**Walkthrough:**

1. `price = 100` must come first, because it creates the starting value.
2. `shipping = price * 0.20` can now run, because `price` exists.
3. `first_subtotal = price + shipping` can now run, because both `price` and `shipping` exist.
4. `first_total = first_subtotal * 1.25` can now run, because `first_subtotal` exists.
5. `first_label = "First total: " + str(first_total)` can now run, because `first_total` exists.
6. `extra_item = len(first_label)` can now run, because `first_label` exists.
7. `second_subtotal = first_total + extra_item` can now run, because both `first_total` and `extra_item` exist.
8. `second_total = second_subtotal * 1.25` can now run, because `second_subtotal` exists.
9. `second_label = first_label + " | Second total: " + str(second_total)` can now run, because both `first_label` and `second_total` exist.
10. `print(second_label)` comes last, because it depends on `second_label`.

**Takeaway:** This task is still about execution order, but the ordered code also contains meaningful smaller steps. The first part builds a subtotal, a total, and a label. The second part then builds another subtotal, total, and label from earlier results. The next tasks focus more directly on identifying this kind of program structure.""",
)


# Concept 2: Abstraction and Decomposition

These tasks focus on how the program is structured, and figuring out where abstractions would make sense. You will group code lines into parts and identify repeated logic that could be abstracted into functions.

The code lines are now in-order.

### Task 2.1: Group lines into blocks

Below is a small program. Divide the lines into **3 logical blocks**:

1. setup  
2. processing  
3. output  

Write the groups in this format:

`1,2; 4,5; 6,7`

### Code

1. `prices = [20, 35, 15]`  
2. `discount = 0.8`  
3. `discounted_prices = []`  
4. `for price in prices:`  
5. `    discounted_prices.append(price * discount)`  
6. `total = sum(discounted_prices)`  
7. `print(total)`

In [11]:

create_block_task(
    correct_blocks=[['1', '2', '3'], ['4', '5', '6'], ['7']],
    all_lines=['1', '2', '3', '4', '5', '6', '7'],
    hint_md="""**Hint:** Look for the boundary between preparation, calculation, and displaying the result.""",
    solution_md="""**Solution:** `1,2,3; 4,5,6; 7`""",
    walkthrough_md="""**Walkthrough:**

- Lines 1-3 are setup: they create the input data, the discount value, and an empty list for the calculated prices.
- Lines 4-6 are processing: the discount is applied, and `total = sum(discounted_prices)` finishes the calculation by summing over values.
- Line 7 is output: it displays the final result.

**Takeaway:** A program can be understood as meaningful parts, not only as individual lines.""",
)


### Small example before Task 2.2

This is not part of the task. It only shows what kind of answer is expected.

Task 2.2 is about finding repeated code, that could be abstracted into a function.

The code below calculates and prints the area of two rectangles.

1. `window_width = 4`  
2. `window_height = 3`  
3. `window_area = window_width * window_height`  
4. `print(area)`  
5. `door_width = 2`  
6. `door_height = 5`  
7. `door_area = door_width * door_height`  
8. `print(door_area)`

One reasonable answer (and the expected input format for the next task) would be:

`Repeated lines: 1,2,3 and 5,6,7`  
`Function name: calculate_area`  
`Function arguments: width, height`

This answer means that the repeated calculation could be moved into a function. The `print(...)` lines are not included, because printing is a separate action from calculating the area.

### Task 2.2: Repeated structure and abstraction

Below is a small Python program, which creates two price labels.

Your task is to identify which lines could be moved into a function that creates a price label for one product.

Do not include the `print(...)` lines in the function. See the example above for the expected input format.

### Code

1. `book_title = "Python 101"`  
2. `book_price = 120`  
3. `book_label = book_title + ": " + str(book_price) + " kr."`  
4. `print(book_label)`  
5. `lamp_name = "Desk lamp"`  
6. `lamp_price = 80`  
7. `lamp_label = lamp_name + ": " + str(lamp_price) + " kr."`  
8. `print(lamp_label)`

In [12]:

create_abstraction_task(
    expected_group_alternatives=[
        [('1', '2', '3'), ('5', '6', '7')],
    ],
    hint_md="""**Hint:** Compare the two repeated sections: which lines have the same purpose, and which values are different between them? A good function name should describe the shared purpose, and the inputs should be the values that change.""",
    solution_md="""**One strong answer:**

One useful function boundary is lines 1,2,3 and 5,6,7.

These lines do the same kind of work for each product:
- store the product name,
- store the product price,
- build a label string.

A possible function could be:

```python
def make_price_label(name, price):
    return name + ": " + str(price) + " kr."
```

The function needs two inputs: the product name and the product price.""",
    walkthrough_md="""**Walkthrough:**

The two blocks are not identical, but they have the same purpose.

Lines 1-3 create a label for a book:

1. define a product title;
2. define a price;
3. combine the title and price into a text label.

Lines 5-7 do the same kind of work for a lamp:

1. define a product name;
2. define a price;
3. combine the name and price into a text label.

A useful abstraction would separate the general label-building idea from the specific products.

The important point is not the exact function name. The important point is noticing that `book_title` and `lamp_name` play the same role, and that `book_price` and `lamp_price` play the same role."""
)


## End

Thank you. The facilitator will now ask a few follow-up questions.